# FlowVAE -- pipeline completo en Colab

Replica TODO el pipeline vigente, desde la busqueda de hiperparametros con Optuna hasta la comparacion final
VAE+SPE vs. AE+SPE vs. baselines convencionales, usando los scripts YA ESCRITOS del repo
(no se reimplementa nada acá adentro -- cada celda solo llama al .py correspondiente via `!python3`,
así el notebook nunca queda desincronizado del código real como pasó con `colab_train_vae.ipynb`).

**Prerrequisito:** `processed/` (train.npz, test.npz, metadata.json, vocabs.pkl, scaler.pkl) generado por
`preprocess.py` y las carpetas de captura de malware, YA subidos a Google Drive. Este notebook NO corre
`preprocess.py` -- arranca directamente en Optuna.

**Pasos (en orden):**
1. Optuna: búsqueda de hiperparámetros (`optuna_search.py`)
2. Entrenamiento final del FlowVAE con los mejores hiperparámetros (`train_vae.py`)
3. Fase 1: calibración de las cartas de control T2/SPE sobre benigno (`control_charts/phase1.py`)
4. Fase 2: cartas de control por familia de ataque (`control_charts/phase2.py`)
5. Matriz de confusión + ROC/AUC/PR-AUC del FlowVAE, T2 y SPE (`control_charts/classification_metrics.py`)
6. Baselines entrenados SOLO con benigno: Isolation Forest, One-Class SVM, LOF, PCA, Autoencoder, Deep SVDD,
   VAE convencional (`baselines/classification_metrics.py`)
7. AE estándar con T2+SPE, mismo criterio UCL que el FlowVAE (`baselines/ae_control_chart.py`)
8. Modelos supervisados entrenados con benigno+maligno mezclado: Random Forest, XGBoost, LightGBM, red neuronal
   (`baselines/supervised_classification_metrics.py`)
9. Tablas de comparación consolidadas (`baselines/build_comparison_tables.py`)
10. Gráficos finales: ROC por ronda + barras de AUC (`baselines/plot_classification_comparison.py`)

**Tiempo estimado:** los pasos 1-3 y 6-10 son rápidos (el dataset benigno de entrenamiento es chico, ~1800 filas).
Los pasos 4 y 5 son los lentos -- recorren la POBLACIÓN COMPLETA de cada familia de ataque (hasta ~11M filas
por familia); contá con **1-2 horas** la primera vez. Ambos cachean resultados por fila (`control_charts/output/scores/`),
así que si se corta la sesión de Colab y se vuelve a correr, no repiten el trabajo ya hecho.

## 0. Setup: Drive, dependencias, paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajustá esto a dónde subiste el proyecto dentro de tu Drive
PROJECT_DIR = '/content/drive/MyDrive/Modelo_VAE'
import os
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

In [ ]:
!pip install pyro-ppl optuna xgboost lightgbm -q

In [ ]:
import os

# Ajustá esto a dónde subiste las capturas de malware dentro de tu Drive
# (por defecto, en la maquina local, common.py apunta a una carpeta FUERA del repo --
# ver control_charts/common.py, MALWARE_ROOT ya lee esta variable de entorno si esta seteada)
os.environ['MALWARE_ROOT'] = f'{PROJECT_DIR}/Capturas_malignas/Malware/capturas_csv'
print('MALWARE_ROOT =', os.environ['MALWARE_ROOT'])

### Verificación de datos preprocesados
Si esto falla, corré `preprocess.py` localmente y subí la carpeta `processed/` a Drive antes de seguir.

In [ ]:
from pathlib import Path

required = ['processed/train.npz', 'processed/test.npz', 'processed/metadata.json',
            'processed/vocabs.pkl', 'processed/scaler.pkl']
missing = [f for f in required if not Path(f).exists()]
assert not missing, f'Faltan estos archivos preprocesados: {missing}'
print('OK: processed/ completo')

malware_root = Path(os.environ['MALWARE_ROOT'])
assert malware_root.exists(), f'No existe MALWARE_ROOT: {malware_root} -- ajustá la celda anterior'
print('OK: MALWARE_ROOT tiene', len(list(malware_root.iterdir())), 'carpetas de ataque')

## 1. Optuna: búsqueda de hiperparámetros del FlowVAE
`z_dim < hidden_dim < input_dim`, más `beta`, `lr`, `batch_size` -- ver docstring de `optuna_search.py`.
Guarda `checkpoints/optuna_best_params.json` y `checkpoints/optuna_trials.csv`.

In [ ]:
N_TRIALS = 100
EPOCHS_OPTUNA = 100
!python3 optuna_search.py --n-trials {N_TRIALS} --epochs {EPOCHS_OPTUNA}

In [ ]:
import json

with open('checkpoints/optuna_best_params.json') as f:
    best = json.load(f)
print(json.dumps(best, indent=2))

## 2. Entrenar el FlowVAE final con los mejores hiperparámetros
Guarda `checkpoints/vae_best.pt` (el que usan TODOS los pasos siguientes), `vae_last.pt`,
`loss_history.csv` y `loss_curve.png`.

In [ ]:
EPOCHS_FINAL = 300  # la busqueda de Optuna uso pocas epochs por trial (poda temprana); el entrenamiento final puede usar mas
p = best['params']
!python3 train_vae.py --epochs {EPOCHS_FINAL} --batch-size {p['batch_size']} --hidden-dim {p['hidden_dim']} --z-dim {p['z_dim']} --beta {p['beta']} --lr {p['lr']}

In [ ]:
from IPython.display import Image, display
display(Image('checkpoints/loss_curve.png'))

## 3. Fase 1 -- calibración de las cartas de control (T2 y SPE) sobre benigno
UCL por bootstrap de bloques móviles, SOLO con `train.npz`. Guarda `control_charts/output/phase1_reference.npz`.

In [ ]:
!python3 control_charts/phase1.py

In [ ]:
display(Image('control_charts/output/phase1_T2.png'))
display(Image('control_charts/output/phase1_SPE.png'))

## 4. Fase 2 -- cartas de control por familia de ataque
Corre las 12 familias (sin `--attack`, procesa todas). **Lento** (población completa, CSVs de varios GB).

In [ ]:
!python3 control_charts/phase2.py

In [ ]:
import pandas as pd
pd.read_csv('control_charts/output/phase2_summary.csv')

## 5. FlowVAE -- matriz de confusión, ROC, AUC, PR-AUC (T2 y SPE)
Benigno test + cada familia de ataque, MEZCLADOS. Cachea el score por fila en `control_charts/output/scores/`
(si se corta la sesión, al volver a correr NO repite el encode de los CSVs ya scoreados). **Lento** la primera vez.

In [ ]:
!python3 control_charts/classification_metrics.py

In [ ]:
display(Image('control_charts/output/confusion_matrix_T2.png'))
display(Image('control_charts/output/confusion_matrix_SPE.png'))
display(Image('control_charts/output/roc_curves.png'))
pd.read_csv('control_charts/output/classification_metrics.csv')

## 6. Baselines entrenados SOLO con benigno
Isolation Forest, One-Class SVM, LOF, PCA, Autoencoder, Deep SVDD, VAE convencional -- umbral = mejor F1,
comparación "pura" vía ROC/AUC. Muestrea hasta 200k+5k filas por familia (cachea en `baselines/output/eval_cache/`).

In [ ]:
!python3 -m baselines.classification_metrics --eval-n 200000 --rf-train-n 5000

## 7. AE estándar -- T2 + SPE con el mismo criterio UCL que el FlowVAE
El comparable directo de la Sección 5 pero con un autoencoder convencional en vez del FlowVAE.
Reusa las muestras cacheadas en el paso 6 (rápido).

In [ ]:
!python3 -m baselines.ae_control_chart

## 8. Modelos supervisados -- entrenados con benigno + maligno MEZCLADO
Random Forest, XGBoost, LightGBM, red neuronal -- umbral nativo (P(ataque) > 0.5), no hay calibración.

In [ ]:
!python3 -m baselines.supervised_classification_metrics

## 9. Tablas de comparación consolidadas
Junta todo en 3 archivos con el MISMO esquema de columnas:
- `comparison_VAE_vs_AE.csv` -- VAE_T2, VAE_SPE, AE_T2, AE_SPE
- `comparison_benign_only_models.csv` -- los 4 anteriores + los 7 baselines benigno-only (11 en total)
- `comparison_mixed_trained_models.csv` -- Random Forest, XGBoost, LightGBM, red neuronal

In [ ]:
!python3 -m baselines.build_comparison_tables

## 10. Gráficos finales: ROC por ronda + ranking de AUC

In [ ]:
!python3 -m baselines.plot_classification_comparison

In [ ]:
display(Image('baselines/output/roc_curves_all_methods.png'))
display(Image('baselines/output/auc_f1_comparison.png'))
pd.read_csv('baselines/output/final_comparison_summary.csv').sort_values('auc', ascending=False)

## Listo
Todos los resultados quedaron en Drive (`control_charts/output/`, `baselines/output/`) --
no hace falta descargar nada, persisten entre sesiones de Colab mientras Drive siga montado con la misma cuenta.